# 03 — Risco de não atingir a meta de 2025 por município

Modelo B: features do município em 2023 → resultado de 2024 (treino); features de 2024 → estimativa de 2025 contra `meta_alfabetizacao_2025` (aplicação). Grão: município, rede municipal. Lê `reports/`; não retreina (`python -m src.modeling.risco_municipio`).

In [1]:
import sys; sys.path.insert(0, "..")
import json
import pandas as pd

from src import config
from src.visualization import plots

pd.set_option("display.width", 160)
mb = json.loads((config.REPORTS / "metricas_modelo_b.json").read_text(encoding="utf-8"))
rank = pd.read_csv(config.REPORTS / "ranking_risco_municipios.csv")
print({k: mb[k] for k in ["n_treino", "n_com_meta", "n_indistinguivel_2024", "n_aplicacao_2024", "n_sem_meta_2025", "n_acima_da_margem", "prob_media_nao_atingir_2025"]})

{'n_treino': 4775, 'n_com_meta': 4775, 'n_indistinguivel_2024': 2040, 'n_aplicacao_2024': 5452, 'n_sem_meta_2025': 100, 'n_acima_da_margem': 1267, 'prob_media_nao_atingir_2025': 0.20746933142862917}


C:\Users\tcarm\Projetos\predicao-alfabetiza-brasil\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Avaliação (RepeatedKFold 5×3, 2023 → 2024)

In [2]:
aval = pd.DataFrame(mb["avaliacao"])
display(aval.pivot_table(index=["tarefa", "modelo"], columns="metrica", values="media").round(3))

metrica                     brier     mae  pr_auc     r2    rmse  roc_auc
tarefa        modelo                                                     
classificacao hgb_clf       0.132     NaN   0.636    NaN     NaN    0.836
              logistica     0.125     NaN   0.630    NaN     NaN    0.843
regressao     hgb_reg         NaN   8.601     NaN  0.656  11.671      NaN
              persistencia    NaN  12.864     NaN    NaN     NaN      NaN
              ridge           NaN   8.704     NaN  0.657  11.656      NaN

Leitura: a **persistência** (prever que 2024 = 2023) é um baseline forte, e a diferença para o HGB mede o que o contexto acrescenta além de "onde o município já estava". A classificação de `nao_atingiu` tem AUC alto porque a taxa atual é muito informativa da meta — o valor do modelo está no `gap` previsto, não na classe. `indistinguivel` (gap dentro da margem de erro) conta como "atingiu" no treino e é reportado à parte.

## 2. Ranking: risco × volume

In [3]:
cols = ["nome_municipio", "sigla_uf", "taxa_2024", "ic95_2024", "criancas_nao_alfabetizadas_2024", "meta_2025", "taxa_prevista_2025", "gap_previsto_2025", "prob_nao_atingir_2025", "acima_da_margem", "prioridade"]
display(rank[cols].head(20).round(2))
plots.salvar(plots.plot_ranking_risco(rank, top=20), "ranking_risco_top20")

,nome_municipio,sigla_uf,taxa_2024,ic95_2024,criancas_nao_alfabetizadas_2024,meta_2025,taxa_prevista_2025,gap_previsto_2025,prob_nao_atingir_2025,acima_da_margem,prioridade
0,Rio de Janeiro,RJ,63.76,0.46,18058.0,63.97,67.35,3.38,0.59,False,10583.04
1,Manaus,AM,50.14,0.73,12280.0,61.28,47.83,-13.45,0.85,True,10457.87
2,Salvador,BA,36.75,0.86,7661.0,52.03,37.07,-14.96,0.88,True,6728.83
3,Curitiba,PR,65.96,0.82,5403.0,73.41,64.69,-8.72,0.88,True,4730.10
4,Duque de Caxias,RJ,45.38,1.21,4510.0,57.80,43.22,-14.58,0.95,True,4292.83
5,Nova Iguaçu,RJ,33.42,1.16,5187.0,47.44,38.09,-9.35,0.62,True,3237.23
6,São Gonçalo,RJ,43.72,1.48,3186.0,56.76,42.20,-14.56,0.97,True,3089.38
7,Campo Grande,MS,51.76,0.99,5361.0,53.84,58.09,4.25,0.57,False,3053.82
8,Recife,PE,54.36,0.99,4911.0,68.08,57.32,-10.76,0.60,True,2964.54
9,São Paulo,SP,48.26,0.48,25407.0,51.06,58.61,7.55,0.11,False,2911.76


WindowsPath('C:/Users/tcarm/Projetos/predicao-alfabetiza-brasil/images/ranking_risco_top20.png')

In [4]:
d = rank.dropna(subset=["meta_2025"]).assign(situacao=lambda r: r["acima_da_margem"].map({True: "abaixo da meta além da margem", False: "dentro da margem / acima"}))
plots.salvar(plots.plot_dispersao(d, "taxa_2024", "taxa_prevista_2025", hue="situacao"), "risco_dispersao")
print(f"municípios previstos abaixo da meta 2025 além do ic95: {int(rank['acima_da_margem'].sum())} de {len(rank)}")
print("por região:", rank.groupby("regiao")["acima_da_margem"].mean().mul(100).round(1).to_dict())

municípios previstos abaixo da meta 2025 além do ic95: 1267 de 5452
por região: {'Centro-Oeste': 4.3, 'Nordeste': 35.1, 'Norte': 33.1, 'Sudeste': 4.2, 'Sul': 35.6}


## 3. Como ler (e como não ler)

- **Prioridade = probabilidade × crianças não alfabetizadas.** Município pequeno com taxa péssima não lidera a lista; capital com taxa mediana e milhares de crianças fora, sim. É o achado 2 da Fase 2 aplicado ao futuro.
- **`acima_da_margem`** exige que o gap previsto supere o `ic95` de 2024: não sinaliza município cuja "queda" cabe no ruído amostral.
- **Regressão à média:** quem subiu muito em 2024 tende a recuar em 2025. O modelo aprende isso da transição 2023 → 2024 — a única treinável. Com a onda de 2025 o modelo pode ser validado de verdade.
- **É risco, não culpa.** O ranking diz onde o esforço tem mais chance de mudar o número, não quem "fez errado".